In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ==============================================================================
# 1. ĐỌC DỮ LIỆU, MÔ TẢ CẤU TRÚC & THỐNG KÊ SƠ BỘ
# ==============================================================================
df = pd.read_csv("/content/drive/MyDrive/BTL_ML/NGUYENTHIDIEULINH_Nhóm số 24_HotelCancelledPredict/data/raw/hotel_bookings.csv")

print("=== 1. THÔNG TIN CẤU TRÚC DỮ LIỆU GỐC ===")
print(f"Số bản ghi (dòng): {df.shape[0]:,}")
print(f"Số trường thông tin (cột): {df.shape[1]}")
print("\nKiểu dữ liệu và số giá trị khuyết thiếu:")
missing_info = pd.DataFrame({
    "Data_Type": df.dtypes,
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_info[missing_info["Missing_Count"] > 0])

print("\nTỷ lệ phân phối biến mục tiêu (is_canceled):")
print((df["is_canceled"].value_counts(normalize=True) * 100).round(2))

print("\nThống kê mô tả các biến số chính trước xử lý:")
print(df[["lead_time", "adr", "adults", "children", "stays_in_week_nights"]].describe().T[["count", "mean", "std", "min", "50%", "max"]])

# ==============================================================================
# 2. XỬ LÝ DỮ LIỆU LỖI (THIẾU TRƯỜNG, RÒ RỈ DỮ LIỆU, NGOẠI LAI, TRÙNG LẶP)
# ==============================================================================
df_clean = df.copy()

# 2.1. Loại bỏ các cột gây rò rỉ dữ liệu (Data Leakage) và cột thiếu >94% dữ liệu
cols_to_drop = ["reservation_status", "reservation_status_date", "company"]
df_clean.drop(columns=cols_to_drop, inplace=True)

# 2.2. Xử lý giá trị thiếu (Missing Values)
df_clean["children"] = df_clean["children"].fillna(0).astype(int)
df_clean["country"] = df_clean["country"].fillna("Unknown")
df_clean["agent"] = df_clean["agent"].fillna(0).astype(int)

# 2.3. Loại bỏ các bản ghi vô lý & giá trị ngoại lai (Outliers)
# Loại bỏ đơn đặt phòng có 0 người lớn, 0 trẻ em, 0 em bé
df_clean = df_clean[(df_clean["adults"] + df_clean["children"] + df_clean["babies"]) > 0]

# Loại bỏ giá phòng (adr) âm hoặc cực đại bất thường (> 1000 EUR/đêm)
df_clean = df_clean[(df_clean["adr"] >= 0) & (df_clean["adr"] <= 1000)]

# 2.4. Loại bỏ các bản ghi trùng lặp hoàn toàn để tránh sai lệch mật độ khi phân cụm/huấn luyện
df_clean.drop_duplicates(inplace=True)
df_clean.reset_index(drop=True, inplace=True)

print(f"\n=== 2. SAU KHI LÀM SẠCH DỮ LIỆU LỖI ===")
print(f"Kích thước dữ liệu sạch: {df_clean.shape[0]:,} dòng x {df_clean.shape[1]} cột")
print(f"Tổng số giá trị NaN còn lại: {df_clean.isnull().sum().sum()}")

# ==============================================================================
# 3. CHUYỂN ĐỔI DỮ LIỆU VỀ DẠNG SỐ PHÙ HỢP (ENCODING)
# ==============================================================================
# 3.1. Mã hóa nhị phân và thứ tự tự nhiên
df_clean["hotel"] = df_clean["hotel"].map({"Resort Hotel": 0, "City Hotel": 1})

month_map = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}
df_clean["arrival_date_month"] = df_clean["arrival_date_month"].map(month_map)

# 3.2. Xử lý biến có nhiều nhãn (High Cardinality):
# - Chuyển 'agent' thành cờ nhị phân 'has_agent' (1: đặt qua đại lý, 0: đặt trực tiếp)
df_clean["has_agent"] = (df_clean["agent"] > 0).astype(int)
df_clean.drop(columns=["agent"], inplace=True)

# - Với 'country' (177 quốc gia): Giữ Top 10 quốc gia phổ biến nhất, gộp còn lại thành 'Other'
top_10_countries = df_clean["country"].value_counts().nlargest(10).index
df_clean["country"] = df_clean["country"].where(df_clean["country"].isin(top_10_countries), "Other")

# 3.3. One-Hot Encoding cho toàn bộ các biến phân loại danh định còn lại
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True, dtype=int)

# ==============================================================================
# 4. CHUẨN HÓA GIÁ TRỊ (STANDARD SCALER) & MÔ TẢ DỮ LIỆU SAU CHUẨN HÓA
# ==============================================================================
# Tách riêng tập đặc trưng X và biến mục tiêu y (giữ nguyên y là 0 và 1)
target_col = "is_canceled"
X = df_encoded.drop(columns=[target_col])
y = df_encoded[target_col]

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

# Ghép lại với cột nhãn y ở cuối cùng để tiện lưu trữ và sử dụng cho các bước sau
df_final_unscaled = pd.concat([X, y], axis=1)
df_final_scaled = pd.concat([X_scaled, y], axis=1)

print("\n=== 3. MÔ TẢ DỮ LIỆU SAU MÃ HÓA VÀ CHUẨN HÓA ===")
print(f"Kích thước ma trận đầu vào X: {X_scaled.shape[0]:,} dòng x {X_scaled.shape[1]} đặc trưng số")
print("Kiểm tra phân phối sau StandardScaler (5 đặc trưng tiêu biểu):")
print(X_scaled[["lead_time", "adr", "adults", "stays_in_week_nights", "total_of_special_requests"]].describe().T[["mean", "std", "min", "50%", "max"]].round(4))

os.makedirs("/content/drive/MyDrive/BTL_ML/NGUYENTHIDIEULINH_Nhóm số 24_HotelCancelledPredict/data/processed", exist_ok=True)
df_final_unscaled.to_csv("/content/drive/MyDrive/BTL_ML/NGUYENTHIDIEULINH_Nhóm số 24_HotelCancelledPredict/data/processed/clean_bookings_unscaled.csv", index=False)
df_final_scaled.to_csv("/content/drive/MyDrive/BTL_ML/NGUYENTHIDIEULINH_Nhóm số 24_HotelCancelledPredict/data/processed/clean_bookings_scaled.csv", index=False)

=== 1. THÔNG TIN CẤU TRÚC DỮ LIỆU GỐC ===
Số bản ghi (dòng): 119,390
Số trường thông tin (cột): 32

Kiểu dữ liệu và số giá trị khuyết thiếu:
         Data_Type  Missing_Count  Missing_Percent (%)
children   float64              4                 0.00
country     object            488                 0.41
agent      float64          16340                13.69
company    float64         112593                94.31

Tỷ lệ phân phối biến mục tiêu (is_canceled):
is_canceled
0    62.96
1    37.04
Name: proportion, dtype: float64

Thống kê mô tả các biến số chính trước xử lý:
                         count        mean         std   min     50%     max
lead_time             119390.0  104.011416  106.863097  0.00  69.000   737.0
adr                   119390.0  101.831122   50.535790 -6.38  94.575  5400.0
adults                119390.0    1.856403    0.579261  0.00   2.000    55.0
children              119386.0    0.103890    0.398561  0.00   0.000    10.0
stays_in_week_nights  119390.0    2.500

## 2.1. Phân tích và Nhận xét Dữ liệu sau Tiền xử lý

### 1. Đánh giá dữ liệu sau khi làm sạch lỗi

Sau khi loại bỏ các trường gây rò rỉ dữ liệu (`reservation_status`, `reservation_status_date`), trường quá thưa (`company`), điền khuyết và lọc bỏ các bản ghi vô lý hoặc trùng lặp, tập dữ liệu thu được có quy mô **$86.962$ bản ghi $\times 29$ thuộc tính** với **$0$ giá trị khuyết thiếu (`NaN`)**.

* **Sự thay đổi phân phối của biến mục tiêu (`is_canceled`):**
  * Ở dữ liệu thô ban đầu, tỷ lệ hủy phòng chiếm **$37,04\%$**. Sau khi loại bỏ $31.994$ bản ghi trùng lặp (vốn tập trung nhiều ở các đơn đặt phòng theo đoàn bị hủy hàng loạt), tỷ lệ hủy thực tế trên các đơn đặt phòng độc lập là **$27,31\%$** ($23.749$ đơn hủy) và không hủy là **$72,69\%$** ($63.213$ đơn thành công).
  * **Ý nghĩa cho bài toán:** Dữ liệu có sự mất cân bằng lớp ở mức trung bình (tỷ lệ xấp xỉ $2,66 : 1$). Ở bước phân loại (Mục 2.4), khi đánh giá hiệu năng mô hình cần kết hợp thêm các chỉ số nhạy với lớp thiểu số như `F1-Score`, `Precision`, `Recall` và `ROC-AUC` thay vì chỉ dựa vào `Accuracy`.
* **Tính toàn vẹn và chất lượng thông tin:**
  * Toàn bộ các đơn đặt phòng không có người ở (`adults + children + babies = 0`), giá phòng âm ($-6,38\text{ EUR}$) và ngoại lai cực đại ($5.400\text{ EUR/đêm}$) đã được loại bỏ hoàn toàn. Giá phòng trung bình thực tế (`adr`) sau làm sạch đạt **$106,49\text{ EUR/đêm}$** (trung vị **$98,27\text{ EUR/đêm}$**, dao động trong khoảng hợp lệ từ $0,0$ đến $510,0\text{ EUR/đêm}$).


### 2. Cấu trúc ma trận đặc trưng sau mã hóa

Từ $28$ biến đầu vào sau làm sạch (tách riêng biến mục tiêu `is_canceled`), quá trình chuyển đổi dữ liệu đã tạo ra ma trận đặc trưng hoàn toàn dạng số kích thước $X \in \mathbb{R}^{86962 \times 68}$:

* **Nhóm biến định lượng giữ nguyên ($18$ đặc trưng):** Các biến đếm và đo lường liên tục như `lead_time`, `adr`, `adults`, `children`, `babies`, `stays_in_weekend_nights`, `stays_in_week_nights`, `previous_cancellations`, `booking_changes`, `days_in_waiting_list`, `required_car_parking_spaces`, `total_of_special_requests`, v.v.
* **Nhóm biến nhị phân và thứ tự ($3$ đặc trưng):** `hotel` ($0$: Resort Hotel, $1$: City Hotel), `arrival_date_month` (mã hóa từ $1$ đến $12$ theo trình tự thời gian), và `has_agent` (chuyển đổi từ cột `agent` để phân biệt khách đặt qua đại lý hay đặt trực tiếp).
* **Nhóm biến giả từ One-Hot Encoding ($47$ đặc trưng):** Sinh ra từ $8$ biến phân loại danh định (`meal`, `country` đã gom nhóm Top 10 + Other, `market_segment`, `distribution_channel`, `reserved_room_type`, `assigned_room_type`, `deposit_type`, `customer_type`) với tùy chọn `drop_first=True` nhằm tránh hiện tượng đa cộng tuyến hoàn hảo (*Dummy Variable Trap*).



### 3. Phân tích phân phối thống kê sau chuẩn hóa (`StandardScaler`)

Phép chuẩn hóa Z-score với công thức $z = \frac{x - \mu}{\sigma}$ đã đưa toàn bộ $68$ đặc trưng về cùng một hệ quy chiếu chuẩn với **giá trị trung bình $\mu = 0,0$** và **độ lệch chuẩn $\sigma = 1,0$**.

| Đặc trưng | Giá trị thực tế trước scale<br>*(Mean / Median / Max)* | Giá trị sau chuẩn hóa Z-score<br>*(Min / Median 50% / Max)* | Đặc điểm phân phối & Nhận xét chuyên sâu |
| :--- | :---: | :---: | :--- |
| **`lead_time`**<br>*(Số ngày đặt trước)* | $79,81$ / $49,00$ / $737,0$ | **$-0,9285$** / **$-0,3584$** / **$7,6461$** | **Phân phối lệch phải (Right-skewed):** Giá trị trung vị ($-0,3584$) nhỏ hơn trung bình ($0,0$), cho thấy hơn $50\%$ khách hàng đặt phòng sớm dưới $49$ ngày. Điểm cực đại nằm ở mức $+7,65\sigma$ ($737$ ngày). |
| **`adr`**<br>*(Giá phòng trung bình)* | $106,49$ / $98,27$ / $510,0$ | **$-2,0516$** / **$-0,1583$** / **$7,7743$** | **Phân phối gần đối xứng và ổn định:** Trung vị ($-0,1583$) nằm rất sát tâm $0,0$. Biên độ trải đều từ $-2,05\sigma$ (phòng khuyến mãi/miễn phí $0\text{ EUR}$) đến $+7,77\sigma$ (hạng phòng cao cấp $510\text{ EUR/đêm}$). |
| **`adults`**<br>*(Số lượng người lớn)* | $1,88$ / $2,00$ / $55,0$ | **$-3,0217$** / **$0,1938$** / **$85,4033$** | **Phân phối tập trung hẹp với đuôi dài (Heavy-tailed):** Phần lớn các đơn tập trung quanh mức $2$ người lớn (trung vị $+0,1938\sigma$), nhưng tồn tại các đơn đặt đoàn lớn lên tới $55$ người tạo điểm cực trị $+85,40\sigma$. |
| **`stays_in_week_nights`**<br>*(Số đêm ngày thường)* | $2,62$ / $2,00$ / $50,0$ | **$-1,2877$** / **$-0,3062$** / **$23,2486$** | **Phân phối lệch phải:** Hơn một nửa số đơn có thời gian lưu trú từ $2$ đêm trở xuống (trung vị $-0,3062\sigma$). Một số ít khách lưu trú dài hạn tới $50$ đêm tạo độ lệch dương $+23,25\sigma$. |
| **`total_of_special_requests`**<br>*(Tổng yêu cầu đặc biệt)* | $0,70$ / $0,00$ / $5,0$ | **$-0,8414$** / **$-0,8414$** / **$5,1660$** | **Phân phối rời rạc lệch về cận dưới:** Giá trị nhỏ nhất và trung vị trùng nhau tại mức **$-0,8414$**, chứng tỏ có trên $50\%$ khách hàng không đưa ra yêu cầu đặc biệt nào (giá trị gốc bằng $0$). |

